# Cross-corpus evaluation — Beat This! on Vienna4x22 & Batik-plays-Mozart

Mazurka-trained Beat This! checkpoints tested **without re-training** on the two out-of-domain corpora packed by [CrossEval.ipynb](CrossEval.ipynb). For my PhD thesis.

- 5 mazurka folds × 2 corpora × 2 ckpt variants (best-val-loss & last)
- Wraps the official `beat_this/beat_this/test.py` CLI — no code duplication, no model surgery
- Env: `beat_this` (see `beat_this/requirements.txt`)

Per-fold output is printed by `test.py` and additionally saved as `beat_this/logs/summary_<name>_S86_F<f>.csv`.

### Vienna4x22
| Fold | Best-val ckpt: Beat F1 | Best-val ckpt: Downbeat F1 | Last ckpt: Beat F1 | Last ckpt: Downbeat F1 |
|-----:|-----------------------:|---------------------------:|-------------------:|-----------------------:|
| 0 | 0.7063 | 0.3320 | 0.6069 | 0.3125 |
| 1 | 0.7176 | **0.3528** | 0.6830 | 0.2813 |
| 2 | 0.7135 | 0.3370 | 0.6653 | 0.2689 |
| 3 | 0.7180 | 0.2441 | **0.6904** | 0.2545 |
| 4 | **0.7259** | 0.3317 | 0.6755 | 0.2251 |
| **Avg ± Std** | **0.7163 ± 0.0064** | **0.3195 ± 0.0385** | **0.6642 ± 0.0298** | **0.2685 ± 0.0289** |
| **Best beat fold = 4** | **0.7259** | **0.3317** | **0.6755** | **0.2251** |

### Batik-plays-Mozart
| Fold | Best-val ckpt: Beat F1 | Best-val ckpt: Downbeat F1 | Last ckpt: Beat F1 | Last ckpt: Downbeat F1 |
|-----:|-----------------------:|---------------------------:|-------------------:|-----------------------:|
| 0 | 0.2935 | **0.1229** | **0.3052** | **0.1352** |
| 1 | 0.2809 | 0.1148 | 0.2664 | 0.1074 |
| 2 | 0.2800 | 0.1201 | 0.2746 | 0.1105 |
| 3 | 0.2920 | 0.1170 | 0.2744 | 0.1018 |
| 4 | **0.2954** | 0.1195 | 0.2771 | 0.1057 |
| **Avg ± Std** | **0.2884 ± 0.0066** | **0.1189 ± 0.0028** | **0.2795 ± 0.0133** | **0.1121 ± 0.0119** |
| **Best beat fold = 4** | **0.2954** | **0.1195** | **0.2771** | **0.1057** |

## 0. Setup — run once

In [1]:
import os, subprocess
from pathlib import Path

# Notebook lives in eval_and_benchmarks/; run BeatThis with its own dir as CWD
BEAT_THIS_DIR = Path.cwd() / "beat_this"
TEST_PY       = BEAT_THIS_DIR / "beat_this" / "test.py"
assert TEST_PY.is_file(), f"BeatThis test.py not found at {TEST_PY}"

# Mazurka-trained ckpts (5 folds, each with best-val + last)
CKPT_BASE = "/media/datadisk/home/22828187/zhanh/202509_icassp_data/workspaces/mazurka_ckpts/ckpt_baselines/beat_this_ckpt/checkpoints/mazurka_h5_22050_S86_F{f}"

# Packed test HDF5s + split CSVs (from CrossEval.ipynb / pack_h5_{vienna,batik})
TEST_WS = "/media/datadisk/home/22828187/zhanh/202509_bark_data/workspaces"
VIENNA = dict(
    name  = "vienna_sr22050",
    h5    = f"{TEST_WS}/hdf5s/vienna_sr22050",
    csv   = f"{TEST_WS}/split_csvs/vienna_sr22050_test.csv",
)
BATIK = dict(
    name  = "batik_sr22050",
    h5    = f"{TEST_WS}/hdf5s/batik_sr22050",
    csv   = f"{TEST_WS}/split_csvs/batik_sr22050_test.csv",
)

def run_test(corpus: dict, folds=range(5), seed=86):
    """Loop 5 mazurka folds; for each, evaluate ckpt-dir of that fold on the corpus.
    --fold is only used by BeatThis for naming the log file; our CSV has no fold column.
    """
    for f in folds:
        cmd = [
            "python", str(TEST_PY),
            "--h5-root",   corpus["h5"],
            "--csv-split", corpus["csv"],
            "--fold",      str(f),
            "--seed",      str(seed),
            "--name",      corpus["name"],
            "--datasplit", "test",
            "--ckpt-dir",  CKPT_BASE.format(f=f),
            "--topk",      "2",
        ]
        subprocess.run(cmd, check=True, cwd=BEAT_THIS_DIR)

---
## 1. Vienna4x22

In [2]:
run_test(VIENNA)

Seed set to 86


Selected 2 checkpoint(s):
  - /media/datadisk/home/22828187/zhanh/202509_icassp_data/workspaces/mazurka_ckpts/ckpt_baselines/beat_this_ckpt/checkpoints/mazurka_h5_22050_S86_F0/mazurka_h5_22050_S86_F0_epoch09-valloss1.2083.ckpt
  - /media/datadisk/home/22828187/zhanh/202509_icassp_data/workspaces/mazurka_ckpts/ckpt_baselines/beat_this_ckpt/checkpoints/mazurka_h5_22050_S86_F0/last.ckpt

==== Evaluating /media/datadisk/home/22828187/zhanh/202509_icassp_data/workspaces/mazurka_ckpts/ckpt_baselines/beat_this_ckpt/checkpoints/mazurka_h5_22050_S86_F0/mazurka_h5_22050_S86_F0_epoch09-valloss1.2083.ckpt ====
Parameters (BeatThis core): trainable=20,251,696 total=20,251,712
Averaged metrics:
  F-measure_beat: 0.7063
  Cemgil_beat: 0.5815
  CMLt_beat: 0.3913
  AMLt_beat: 0.3919
  F-measure_downbeat: 0.3320
  Cemgil_downbeat: 0.2988
  CMLt_downbeat: 0.0055
  AMLt_downbeat: 0.0727

==== Evaluating /media/datadisk/home/22828187/zhanh/202509_icassp_data/workspaces/mazurka_ckpts/ckpt_baselines/beat_thi

Seed set to 86


Selected 2 checkpoint(s):
  - /media/datadisk/home/22828187/zhanh/202509_icassp_data/workspaces/mazurka_ckpts/ckpt_baselines/beat_this_ckpt/checkpoints/mazurka_h5_22050_S86_F1/mazurka_h5_22050_S86_F1_epoch24-valloss0.8547.ckpt
  - /media/datadisk/home/22828187/zhanh/202509_icassp_data/workspaces/mazurka_ckpts/ckpt_baselines/beat_this_ckpt/checkpoints/mazurka_h5_22050_S86_F1/last.ckpt

==== Evaluating /media/datadisk/home/22828187/zhanh/202509_icassp_data/workspaces/mazurka_ckpts/ckpt_baselines/beat_this_ckpt/checkpoints/mazurka_h5_22050_S86_F1/mazurka_h5_22050_S86_F1_epoch24-valloss0.8547.ckpt ====
Parameters (BeatThis core): trainable=20,251,696 total=20,251,712
Averaged metrics:
  F-measure_beat: 0.7176
  Cemgil_beat: 0.6268
  CMLt_beat: 0.3977
  AMLt_beat: 0.4013
  F-measure_downbeat: 0.3528
  Cemgil_downbeat: 0.3281
  CMLt_downbeat: 0.1124
  AMLt_downbeat: 0.1914

==== Evaluating /media/datadisk/home/22828187/zhanh/202509_icassp_data/workspaces/mazurka_ckpts/ckpt_baselines/beat_thi

Seed set to 86


Selected 2 checkpoint(s):
  - /media/datadisk/home/22828187/zhanh/202509_icassp_data/workspaces/mazurka_ckpts/ckpt_baselines/beat_this_ckpt/checkpoints/mazurka_h5_22050_S86_F2/mazurka_h5_22050_S86_F2_epoch14-valloss1.1175.ckpt
  - /media/datadisk/home/22828187/zhanh/202509_icassp_data/workspaces/mazurka_ckpts/ckpt_baselines/beat_this_ckpt/checkpoints/mazurka_h5_22050_S86_F2/last.ckpt

==== Evaluating /media/datadisk/home/22828187/zhanh/202509_icassp_data/workspaces/mazurka_ckpts/ckpt_baselines/beat_this_ckpt/checkpoints/mazurka_h5_22050_S86_F2/mazurka_h5_22050_S86_F2_epoch14-valloss1.1175.ckpt ====
Parameters (BeatThis core): trainable=20,251,696 total=20,251,712
Averaged metrics:
  F-measure_beat: 0.7135
  Cemgil_beat: 0.6331
  CMLt_beat: 0.3657
  AMLt_beat: 0.3662
  F-measure_downbeat: 0.3370
  Cemgil_downbeat: 0.3262
  CMLt_downbeat: 0.0835
  AMLt_downbeat: 0.1541

==== Evaluating /media/datadisk/home/22828187/zhanh/202509_icassp_data/workspaces/mazurka_ckpts/ckpt_baselines/beat_thi

Seed set to 86


Selected 2 checkpoint(s):
  - /media/datadisk/home/22828187/zhanh/202509_icassp_data/workspaces/mazurka_ckpts/ckpt_baselines/beat_this_ckpt/checkpoints/mazurka_h5_22050_S86_F3/mazurka_h5_22050_S86_F3_epoch29-valloss1.5129.ckpt
  - /media/datadisk/home/22828187/zhanh/202509_icassp_data/workspaces/mazurka_ckpts/ckpt_baselines/beat_this_ckpt/checkpoints/mazurka_h5_22050_S86_F3/last.ckpt

==== Evaluating /media/datadisk/home/22828187/zhanh/202509_icassp_data/workspaces/mazurka_ckpts/ckpt_baselines/beat_this_ckpt/checkpoints/mazurka_h5_22050_S86_F3/mazurka_h5_22050_S86_F3_epoch29-valloss1.5129.ckpt ====
Parameters (BeatThis core): trainable=20,251,696 total=20,251,712
Averaged metrics:
  F-measure_beat: 0.7180
  Cemgil_beat: 0.6259
  CMLt_beat: 0.4205
  AMLt_beat: 0.4207
  F-measure_downbeat: 0.2441
  Cemgil_downbeat: 0.2330
  CMLt_downbeat: 0.0294
  AMLt_downbeat: 0.0994

==== Evaluating /media/datadisk/home/22828187/zhanh/202509_icassp_data/workspaces/mazurka_ckpts/ckpt_baselines/beat_thi

Seed set to 86


Selected 2 checkpoint(s):
  - /media/datadisk/home/22828187/zhanh/202509_icassp_data/workspaces/mazurka_ckpts/ckpt_baselines/beat_this_ckpt/checkpoints/mazurka_h5_22050_S86_F4/mazurka_h5_22050_S86_F4_epoch14-valloss1.2587.ckpt
  - /media/datadisk/home/22828187/zhanh/202509_icassp_data/workspaces/mazurka_ckpts/ckpt_baselines/beat_this_ckpt/checkpoints/mazurka_h5_22050_S86_F4/last.ckpt

==== Evaluating /media/datadisk/home/22828187/zhanh/202509_icassp_data/workspaces/mazurka_ckpts/ckpt_baselines/beat_this_ckpt/checkpoints/mazurka_h5_22050_S86_F4/mazurka_h5_22050_S86_F4_epoch14-valloss1.2587.ckpt ====
Parameters (BeatThis core): trainable=20,251,696 total=20,251,712
Averaged metrics:
  F-measure_beat: 0.7259
  Cemgil_beat: 0.6512
  CMLt_beat: 0.4009
  AMLt_beat: 0.4014
  F-measure_downbeat: 0.3317
  Cemgil_downbeat: 0.3263
  CMLt_downbeat: 0.0440
  AMLt_downbeat: 0.1578

==== Evaluating /media/datadisk/home/22828187/zhanh/202509_icassp_data/workspaces/mazurka_ckpts/ckpt_baselines/beat_thi

---
## 2. Batik-plays-Mozart

In [3]:
run_test(BATIK)

Seed set to 86


Selected 2 checkpoint(s):
  - /media/datadisk/home/22828187/zhanh/202509_icassp_data/workspaces/mazurka_ckpts/ckpt_baselines/beat_this_ckpt/checkpoints/mazurka_h5_22050_S86_F0/mazurka_h5_22050_S86_F0_epoch09-valloss1.2083.ckpt
  - /media/datadisk/home/22828187/zhanh/202509_icassp_data/workspaces/mazurka_ckpts/ckpt_baselines/beat_this_ckpt/checkpoints/mazurka_h5_22050_S86_F0/last.ckpt

==== Evaluating /media/datadisk/home/22828187/zhanh/202509_icassp_data/workspaces/mazurka_ckpts/ckpt_baselines/beat_this_ckpt/checkpoints/mazurka_h5_22050_S86_F0/mazurka_h5_22050_S86_F0_epoch09-valloss1.2083.ckpt ====
Parameters (BeatThis core): trainable=20,251,696 total=20,251,712
Averaged metrics:
  F-measure_beat: 0.2935
  Cemgil_beat: 0.2716
  CMLt_beat: 0.0363
  AMLt_beat: 0.0992
  F-measure_downbeat: 0.1229
  Cemgil_downbeat: 0.1175
  CMLt_downbeat: 0.0075
  AMLt_downbeat: 0.0265

==== Evaluating /media/datadisk/home/22828187/zhanh/202509_icassp_data/workspaces/mazurka_ckpts/ckpt_baselines/beat_thi

Seed set to 86


Selected 2 checkpoint(s):
  - /media/datadisk/home/22828187/zhanh/202509_icassp_data/workspaces/mazurka_ckpts/ckpt_baselines/beat_this_ckpt/checkpoints/mazurka_h5_22050_S86_F1/mazurka_h5_22050_S86_F1_epoch24-valloss0.8547.ckpt
  - /media/datadisk/home/22828187/zhanh/202509_icassp_data/workspaces/mazurka_ckpts/ckpt_baselines/beat_this_ckpt/checkpoints/mazurka_h5_22050_S86_F1/last.ckpt

==== Evaluating /media/datadisk/home/22828187/zhanh/202509_icassp_data/workspaces/mazurka_ckpts/ckpt_baselines/beat_this_ckpt/checkpoints/mazurka_h5_22050_S86_F1/mazurka_h5_22050_S86_F1_epoch24-valloss0.8547.ckpt ====
Parameters (BeatThis core): trainable=20,251,696 total=20,251,712
Averaged metrics:
  F-measure_beat: 0.2809
  Cemgil_beat: 0.2553
  CMLt_beat: 0.0484
  AMLt_beat: 0.1132
  F-measure_downbeat: 0.1148
  Cemgil_downbeat: 0.1048
  CMLt_downbeat: 0.0251
  AMLt_downbeat: 0.0544

==== Evaluating /media/datadisk/home/22828187/zhanh/202509_icassp_data/workspaces/mazurka_ckpts/ckpt_baselines/beat_thi

Seed set to 86


Selected 2 checkpoint(s):
  - /media/datadisk/home/22828187/zhanh/202509_icassp_data/workspaces/mazurka_ckpts/ckpt_baselines/beat_this_ckpt/checkpoints/mazurka_h5_22050_S86_F2/mazurka_h5_22050_S86_F2_epoch14-valloss1.1175.ckpt
  - /media/datadisk/home/22828187/zhanh/202509_icassp_data/workspaces/mazurka_ckpts/ckpt_baselines/beat_this_ckpt/checkpoints/mazurka_h5_22050_S86_F2/last.ckpt

==== Evaluating /media/datadisk/home/22828187/zhanh/202509_icassp_data/workspaces/mazurka_ckpts/ckpt_baselines/beat_this_ckpt/checkpoints/mazurka_h5_22050_S86_F2/mazurka_h5_22050_S86_F2_epoch14-valloss1.1175.ckpt ====
Parameters (BeatThis core): trainable=20,251,696 total=20,251,712
Averaged metrics:
  F-measure_beat: 0.2800
  Cemgil_beat: 0.2588
  CMLt_beat: 0.0472
  AMLt_beat: 0.1101
  F-measure_downbeat: 0.1201
  Cemgil_downbeat: 0.1107
  CMLt_downbeat: 0.0220
  AMLt_downbeat: 0.0473

==== Evaluating /media/datadisk/home/22828187/zhanh/202509_icassp_data/workspaces/mazurka_ckpts/ckpt_baselines/beat_thi

Seed set to 86


Selected 2 checkpoint(s):
  - /media/datadisk/home/22828187/zhanh/202509_icassp_data/workspaces/mazurka_ckpts/ckpt_baselines/beat_this_ckpt/checkpoints/mazurka_h5_22050_S86_F3/mazurka_h5_22050_S86_F3_epoch29-valloss1.5129.ckpt
  - /media/datadisk/home/22828187/zhanh/202509_icassp_data/workspaces/mazurka_ckpts/ckpt_baselines/beat_this_ckpt/checkpoints/mazurka_h5_22050_S86_F3/last.ckpt

==== Evaluating /media/datadisk/home/22828187/zhanh/202509_icassp_data/workspaces/mazurka_ckpts/ckpt_baselines/beat_this_ckpt/checkpoints/mazurka_h5_22050_S86_F3/mazurka_h5_22050_S86_F3_epoch29-valloss1.5129.ckpt ====
Parameters (BeatThis core): trainable=20,251,696 total=20,251,712
Averaged metrics:
  F-measure_beat: 0.2920
  Cemgil_beat: 0.2692
  CMLt_beat: 0.0383
  AMLt_beat: 0.1085
  F-measure_downbeat: 0.1170
  Cemgil_downbeat: 0.1159
  CMLt_downbeat: 0.0162
  AMLt_downbeat: 0.0431

==== Evaluating /media/datadisk/home/22828187/zhanh/202509_icassp_data/workspaces/mazurka_ckpts/ckpt_baselines/beat_thi

Seed set to 86


Selected 2 checkpoint(s):
  - /media/datadisk/home/22828187/zhanh/202509_icassp_data/workspaces/mazurka_ckpts/ckpt_baselines/beat_this_ckpt/checkpoints/mazurka_h5_22050_S86_F4/mazurka_h5_22050_S86_F4_epoch14-valloss1.2587.ckpt
  - /media/datadisk/home/22828187/zhanh/202509_icassp_data/workspaces/mazurka_ckpts/ckpt_baselines/beat_this_ckpt/checkpoints/mazurka_h5_22050_S86_F4/last.ckpt

==== Evaluating /media/datadisk/home/22828187/zhanh/202509_icassp_data/workspaces/mazurka_ckpts/ckpt_baselines/beat_this_ckpt/checkpoints/mazurka_h5_22050_S86_F4/mazurka_h5_22050_S86_F4_epoch14-valloss1.2587.ckpt ====
Parameters (BeatThis core): trainable=20,251,696 total=20,251,712
Averaged metrics:
  F-measure_beat: 0.2954
  Cemgil_beat: 0.2766
  CMLt_beat: 0.0327
  AMLt_beat: 0.0987
  F-measure_downbeat: 0.1195
  Cemgil_downbeat: 0.1117
  CMLt_downbeat: 0.0143
  AMLt_downbeat: 0.0408

==== Evaluating /media/datadisk/home/22828187/zhanh/202509_icassp_data/workspaces/mazurka_ckpts/ckpt_baselines/beat_thi